# Train and apply a noise-aware broadband SOM

This notebook exposes the steps in `bin/compare_som.py`: load fixed broadband
measurements, train a map, estimate a redshift PDF in each cell, assign test
objects, and evaluate their predictions. It trains one **new** NoiseSOM from
`data/parent.npz`; saved comparison outputs and flow checkpoints are not required.

The defaults reproduce the first seed of the selected NoiseSOM configuration:
32 × 32 cells, 60,000 updates, smoothing width 0.01 in ln(1 + z), and a global
redshift-distribution pseudo-count of five. The complete comparison selected
these settings on validation data and repeated five seeds. This demo keeps the
settings fixed and evaluates one seed; it does not repeat that search or train a flow.

Only degraded CFHT u/HSC grizy fluxes, flux errors, and secure spectroscopic
labels are used. COSMOS PDFs and native deep photometry are never loaded.
This is a **single broadband SOM**, not the complete deep-to-wide SOMPZ pipeline.

Use the project's Conda environment and select its Jupyter kernel (see the root
README). Place the parent at `data/parent.npz`. The commit-pinned NoiseSOM source
and license are downloaded on first use; copy `data/sompz_noise/` too for an
offline run. Training runs on the CPU and may take several minutes.
Save executed notebooks under `results/notebooks_executed/`; keep this source clean.

## 1. Imports and settings

We reuse the source loader, bounded-memory cell assignment, and diagnostics
from `compare_som.py`. The actual map fit and the construction of cell PDFs
are shown below rather than hidden behind its full comparison runner.
Paths and the single-seed settings can be changed in this cell.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'bin/compare_som.py').exists()
)
sys.path.insert(0, str(root / 'bin'))
sys.path.insert(0, str(root / 'src'))
from compare_som import NOISE_COMMIT, NOISE_SHA256, assign, density_at, load_noise_som, score
from pzflow.utils import PDFSampler

parent_path = root / 'data/parent.npz'
source_path = root / 'data/sompz_noise/NoiseSOM.py'
out_dir = root / 'results/noisesom_demo'
seed = 20261024
side = 32
n_updates = 60000
bandwidth = 0.01
alpha = 5.0

# Change out_dir for another saved run; existing models are protected.
if out_dir.exists() and any(out_dir.iterdir()):
    raise FileExistsError(f'Choose an empty output directory: {out_dir}')

plt.rcParams.update(
    {
        'font.family': 'serif',
        'font.size': 11,
        'axes.linewidth': 1.5,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'legend.frameon': False,
    }
)

## 2. Load the parent and select the spectroscopic samples

`flux` and `fluxerr` have shape (N, 6), in microJy, with band order u, g, r, i, z, y.
These are already degraded measurements; **do not add noise again** for this
fixed-photometry control. Negative flux measurements are retained.

`split` assigns spatial cells to training (0), validation (1), and test (2).
A finite `zspec` identifies a secure spectroscopic label. Only training spectra
will populate cell redshift histograms. Test spectra are used for scoring only.

In [ ]:
with np.load(parent_path) as f:
    metadata = json.loads(str(f['metadata']))
    if not metadata.get('canonical_parent'):
        raise ValueError('Use the canonical data/parent.npz')
    data = {key: f[key] for key in ['id', 'flux', 'fluxerr', 'zspec', 'split', 'imag', 'cell']}

with parent_path.open('rb') as f:
    parent_sha256 = hashlib.file_digest(f, 'sha256').hexdigest()

flux = data['flux'].astype(float)
err = data['fluxerr'].astype(float)
zspec = data['zspec']
train, valid, test = [
    np.flatnonzero((data['split'] == split) & np.isfinite(zspec)) for split in range(3)
]

assert flux.shape == err.shape == (len(data['id']), 6)
assert len(np.unique(data['id'])) == len(data['id'])
assert np.isfinite(flux).all() and np.isfinite(err).all() and (err > 0).all()
assert np.all((zspec[train] > 0) & (zspec[train] <= 10))
for a, b in [(train, valid), (train, test), (valid, test)]:
    assert not np.intersect1d(data['cell'][a], data['cell'][b]).size

for name, rows in zip(['Training', 'Validation', 'Test'], [train, valid, test]):
    print(f'{name}: {len(rows):,} galaxies')
print(f'Parent SHA256: {parent_sha256}')

## 3. Train the map on broadband fluxes and errors

A SOM cell contains a prototype six-band flux vector. Training moves the
prototypes toward training galaxies under the published noise-aware distance
metric. This step receives **no redshift labels**.

We use exactly the setup in `train_noise_map`: sample 60,000 training rows with
replacement, fix both random-number seeds, initialize from sampled data, and
use the published neighborhood schedule. The metric uses flux uncertainties
directly; errors are not extra coordinates. The 0.02 fractional-error floor
applies inside map training. Classification below uses the supplied errors,
matching `compare_som.py`.

In [ ]:
noise = load_noise_som(source_path)
metric = noise.AsinhMetric(lnScaleSigma=0.4, lnScaleStep=0.03)

rows = np.random.default_rng(seed).integers(len(train), size=n_updates)
np.random.seed(seed)
start = time.monotonic()
som = noise.NoiseSOM(
    metric,
    flux[train][rows],
    err[train][rows],
    learning=noise.hFunc(n_updates, sigma=(30.0, 1.0)),
    shape=(side, side),
    wrap=False,
    logF=True,
    initialize='sample',
    minError=0.02,
)
weights = som.weights
training_seconds = time.monotonic() - start
assert weights.shape == (side * side, 6)
assert np.isfinite(weights).all() and (weights > 0).all()
print(f'Trained {side} x {side} cells in {training_seconds:.1f} s')

## 4. Assign training galaxies and count their redshifts

For every training galaxy, choose the cell with the smallest noise-aware
distance (the best-matching unit). `assign` evaluates this in ten-galaxy chunks
to avoid a large all-galaxy/all-cell distance array.

Within each cell we count **training spectroscopic redshifts** in 2,000 uniform
bins of t = ln(1 + z), spanning 0 ≤ z ≤ 10. This has approximately constant
resolution in Δz/(1 + z). The learned flux map itself is now fixed.

In [ ]:
train_cell = assign(flux[train], err[train], weights, metric)

edges = np.linspace(0, np.log(11), 2001)
grid = np.expm1(np.r_[0, (edges[1:] + edges[:-1]) / 2, edges[-1]])
zbin = np.clip(np.searchsorted(edges, np.log1p(zspec[train])) - 1, 0, len(edges) - 2)
hist = np.zeros((len(weights), len(edges) - 1))
np.add.at(hist, (train_cell, zbin), 1)
count = hist.sum(axis=1)
global_hist = hist.sum(axis=0) / hist.sum()
assert count.sum() == len(train)
print(f'Occupied cells: {(count > 0).sum():,} / {len(weights):,}')

## 5. Turn cell histograms into continuous redshift PDFs

An empty cell borrows the histogram of the closest populated prototype under
the same metric, using median training errors for the prototype comparison.
No validation or test redshift is used to fill it.

Add `alpha` pseudo-counts drawn from the global training redshift distribution,
then smooth along ln(1 + z) with width `bandwidth`. Divide by (1 + z) to convert
a density in ln(1 + z) into a density in z. `PDFSampler` normalizes this
piecewise-linear density and integrates its CDF. These operations mirror
`cell_pdfs` in the comparison script.

In [ ]:
filled = np.flatnonzero(count > 0)
empty = np.flatnonzero(count == 0)
fallback = np.arange(len(weights))
prototype_err = np.median(err[train], axis=0)
if len(empty):
    nearest = assign(
        weights[empty],
        np.broadcast_to(prototype_err, weights[empty].shape),
        weights[filled],
        metric,
    )
    fallback[empty] = filled[nearest]

hist_pdf = hist[fallback] + alpha * global_hist
hist_pdf = gaussian_filter1d(hist_pdf, bandwidth / np.diff(edges)[0], axis=1, mode='reflect')
hist_pdf /= hist_pdf.sum(axis=1)[:, None]
pdf = np.pad(hist_pdf, ((0, 0), (1, 1)), mode='edge') / (1 + grid)[None, :]
sampler = PDFSampler(grid, pdf)
assert np.allclose(np.trapezoid(sampler.p, grid, axis=1), 1, atol=1e-6)
assert np.all(np.diff(sampler.c, axis=1) >= -1e-7)
assert np.allclose(sampler.c[:, -1], 1)

Inspect the training occupancy and the mean redshift of each cell. The redshift
map comes from the training labels, not from test galaxies. Empty cells use the
fallback PDFs constructed above.

In [ ]:
mean_z = np.trapezoid(sampler.p * grid[None, :], grid, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
im = axes[0].imshow(np.log10(1 + count.reshape(side, side)), origin='lower')
fig.colorbar(im, ax=axes[0], label=r'$\log_{10}(1 + N_{\mathrm{train}})$')
im = axes[1].imshow(mean_z.reshape(side, side), origin='lower', vmin=0, vmax=3)
fig.colorbar(im, ax=axes[1], label=r'Cell mean $z$')
for ax, title in zip(axes, ['Training occupancy', 'Cell redshift distributions']):
    ax.set(xlabel='SOM column', ylabel='SOM row', title=title)
plt.show()

## 6. Check the implementation and validation density

The first check compares the explicit PDF construction above with `cell_pdfs`
from the script. This guards against the notebook drifting from the comparison.
We also reproduce the comparison's first-seed 4,096-object validation subset
and report its negative log density. We do not select settings on test labels.
The defaults were already selected in the full comparison; this is not a new
independent hyperparameter search.

In [ ]:
from compare_som import cell_pdfs

reference, ref_count, ref_fallback = cell_pdfs(
    train_cell, zspec[train], weights, bandwidth, alpha, metric, prototype_err
)
np.testing.assert_array_equal(count, ref_count)
np.testing.assert_array_equal(fallback, ref_fallback)
np.testing.assert_allclose(sampler.p, reference.p, rtol=0, atol=0)
del reference

vi = np.random.default_rng(seed).choice(valid, min(len(valid), 4096), replace=False)
valid_cell = assign(flux[vi], err[vi], weights, metric)
valid_nll = -np.log(np.maximum(density_at(sampler, zspec[vi], valid_cell), 1e-12)).mean()
print(f'Validation NLL: {valid_nll:.4f}')

## 7. Apply the trained map to test photometry

First assign each test galaxy using only its six fluxes and uncertainties.
Its predicted PDF is the calibrated PDF of its assigned cell. Galaxies assigned
to the same cell therefore share a PDF; the flow benchmark does not impose this
discretization. Redshift labels enter only the evaluation cell below.

`q` stores the 2.5%, 16%, 50%, 84%, and 97.5% quantiles in that order.

In [ ]:
test_cell = assign(flux[test], err[test], weights, metric)
levels = np.array([0.025, 0.16, 0.5, 0.84, 0.975])
cell_quantiles = np.array([np.interp(levels, cdf, grid) for cdf in sampler.c])
q = cell_quantiles[test_cell]

# Use the same evaluation grid as compare_som.py.
eval_grid = np.linspace(0, 10, 1001)
cell_cdf = np.array([np.interp(eval_grid, grid, cdf) for cdf in sampler.c])
cdf = cell_cdf[test_cell]
assert np.isfinite(q).all() and np.all(np.diff(q, axis=1) >= 0)
print(f'Predicted {len(test_cell):,} test galaxies')
print(f'Assigned to an empty training cell: {np.mean(count[test_cell] == 0):.2%}')

## 8. Evaluate against held-out spectroscopy

Use posterior medians for point estimates. NMAD measures robust normalized
scatter; outliers have |z_med − z_spec|/(1 + z_spec) > 0.15. Coverage is the
fraction of truths inside equal-tailed intervals. PIT is the predicted CDF
at the spectroscopic redshift. CRPS scores the full redshift distribution
(lower is better). These diagnostics describe the selected spectroscopic
test subset, not calibration of the full photometric population.

In [ ]:
ztest = zspec[test]
pit = sampler.pit(ztest, test_cell)
nll = -np.log(np.maximum(density_at(sampler, ztest, test_cell), 1e-12))
result = score(ztest, q, pit, nll, cdf, eval_grid)
print(json.dumps(result, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
axes[0].hexbin(ztest, q[:, 2], gridsize=60, bins='log', mincnt=1, cmap='viridis')
axes[0].plot([0, 5], [0, 5], 'k--', lw=1)
axes[0].set(xlim=(0, 5), ylim=(0, 5), xlabel=r'$z_{\mathrm{spec}}$', ylabel=r'Median $z$')
credibility = np.linspace(0, 1, 101)
rank = np.abs(2 * pit - 1)
coverage = np.array([(rank <= value).mean() for value in credibility])
coverage[0], coverage[-1] = 0, 1
axes[1].plot(credibility, coverage, label='NoiseSOM')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Nominal')
axes[1].set(xlim=(0, 1), ylim=(0, 1), xlabel='Nominal credibility', ylabel='Test coverage')
axes[1].legend()
plt.show()

Example test PDFs below span the test sample's magnitude distribution. The
vertical line is the spectroscopic redshift, shown for evaluation only.

In [ ]:
order = np.argsort(data['imag'][test])
examples = order[np.linspace(0, len(order) - 1, 6, dtype=int)]
fig, axes = plt.subplots(2, 3, figsize=(10, 5.5), constrained_layout=True)
for ax, j in zip(axes.flat, examples):
    ax.plot(grid, sampler.p[test_cell[j]], label='NoiseSOM PDF')
    ax.axvline(ztest[j], color='k', ls='--', label=r'$z_{\mathrm{spec}}$')
    right = min(10, max(q[j, 4], ztest[j]) + 0.3)
    ax.set(xlim=(0, right), xlabel=r'$z$', ylabel=r'$p(z)$')
    ax.set_title(f'ID {data["id"][test[j]]}; i = {data["imag"][test[j]]:.2f}')
axes.flat[0].legend(fontsize=8)
plt.show()

## 9. Save the trained map, cell PDFs, predictions, and provenance

Outputs go to the local, ignored demo directory selected above. `som.npz`
contains everything needed to reconstruct this trained predictor: the
prototype fluxes, redshift grid, normalized cell PDFs, occupancy, and fallback
indices. `predictions.npz` retains the comparison script's `som_*` field names.
No flow training or cached scientific outputs are modified.

In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    out_dir / 'som.npz',
    weights=weights,
    grid=grid,
    pdf=sampler.p,
    count=count,
    fallback=fallback,
)
prediction = {key: data[key][test] for key in ['id', 'zspec', 'imag', 'cell', 'flux', 'fluxerr']}
prediction.update(
    grid=eval_grid,
    som_quantiles=q,
    som_pit=pit,
    som_nll=nll,
    som_cdf=cdf,
    som_cell=test_cell,
    som_training_occupancy=count[test_cell],
)
np.savez_compressed(out_dir / 'predictions.npz', **prediction)
run = {
    'seed': seed,
    'side': side,
    'iterations': n_updates,
    'bandwidth': bandwidth,
    'alpha': alpha,
    'parent_sha256': parent_sha256,
    'noise_source': {'commit': NOISE_COMMIT, 'sha256': NOISE_SHA256},
    'noise_settings': {
        'lnScaleSigma': 0.4,
        'lnScaleStep': 0.03,
        'minError': 0.02,
        'sigma': [30.0, 1.0],
        'wrap': False,
        'initialize': 'sample',
    },
    'sample_counts': [len(rows) for rows in [train, valid, test]],
    'inputs': 'Fixed degraded six-band fluxes/errors; secure training spectra; no augmentation',
    'validation_nll': float(valid_nll),
    'training_seconds': training_seconds,
    'metrics': result,
}
with (out_dir / 'run.json').open('w') as f:
    json.dump(run, f, indent=2)
    f.write('\n')
print(f'Saved model and predictions to {out_dir}')

## 10. Reload and predict a small batch

This is the application path for an already trained model. Replace the example
batch with new six-band flux/error arrays in the same units and band order.
Spectroscopic redshifts are not required. The pinned source is used only to
construct the same assignment metric; **the map is not retrained**.

In [ ]:
with np.load(out_dir / 'som.npz') as f:
    saved_weights = f['weights']
    saved_sampler = PDFSampler(f['grid'], f['pdf'])

new_flux = flux[test[:5]]
new_err = err[test[:5]]
new_cell = assign(new_flux, new_err, saved_weights, metric)
new_q = np.array(
    [np.interp(levels, saved_sampler.c[cell], saved_sampler.grid) for cell in new_cell]
)
np.testing.assert_array_equal(new_cell, test_cell[:5])
np.testing.assert_allclose(new_q, q[:5], atol=1e-5, rtol=1e-5)
print('Rows: test galaxies; columns: 2.5%, 16%, 50%, 84%, 97.5% redshift quantiles')
print(new_q)